# Quoridor AI — 9×9 Training (N=2, Parallel Self-Play)

**Group 501** | Colman College | DL Final Project

Full-size Quoridor (9×9, 10 walls/player) trained via parallel self-play with
batched GPU inference. Uses `parallel_self_play_mp.py` workers → single GPU
inference thread.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (A100 preferred if available)
2. Run cells in order
3. If Colab disconnects, re-run from **Section 1** — `resume=True` picks up automatically

## Security & Data Warning

**DO NOT commit or upload:**
- API keys, authentication tokens, or passwords
- Model checkpoints containing sensitive training data
- File paths with personal/org information

**Best practices:**
- Store checkpoints in private cloud storage or local machine
- Use `.gitignore` to exclude model files: `*.pth`, `*.ckpt`, `configs/*.local`
- For Colab: download checkpoints before session ends or save to private Drive

## Expected Output & Monitoring

**During training, expect:**
- `[GPU] 500 batches processed...` every ~500 batches (progress indicator)
- Game summaries: `winner=Player X, walls: [8, 9, 8, 10]` (one per iteration)
- Win statistics at iteration end: `{0: 45, 1: 35, 2: 10, None: 10}` (draws)
- Loss/accuracy metrics from model training

**Runtime expectations (9x9, N=2, T4 GPU):**
- ~8 min per self-play iteration (200 games with batched inference)
- ~2 min model training per iteration
- Full training (100 iterations): ~16-20 hours

**Troubleshooting:**
- **OOM error**: Reduce `inference_batch_size` in config or `num_workers`
- **Slow training**: Ensure GPU is active (check `!nvidia-smi`)
- **Hung process**: Cells timeout after 90 min; re-run will resume with `resume=True`

---
## 1. Environment Setup
Run once per Colab session.

In [ ]:
# 1.1 — Install dependencies and setup repo
import os, sys

# Find repo root (works from any starting directory)
REPO_DIR = None
for candidate in [
    os.getcwd(),                             # already in repo root
    os.path.join(os.getcwd(), "dl-quoridor"),  # Jupyter workspace root
    os.path.dirname(os.getcwd()),            # running from notebooks/
]:
    if os.path.exists(os.path.join(candidate, "src", "mcts")):
        REPO_DIR = candidate
        break

assert REPO_DIR is not None, (
    "Could not find dl-quoridor repo. "
    "Make sure the notebook is inside the repo or one level above it."
)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo: {REPO_DIR}")

# Install requirements (--ignore-installed handles system-managed packages)
!pip install -r requirements.txt -q --ignore-installed

import torch
print(f"torch {torch.__version__} installed")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected — training will run on CPU (slow for 9x9).")

In [ ]:
# 1.2 — Set run directory for checkpoints
RUN_DIR = os.path.join(REPO_DIR, "runs", "n2_9x9_v1")
os.makedirs(RUN_DIR, exist_ok=True)
print(f"Run dir: {RUN_DIR}")

---
## 2. Configuration

In [ ]:
# 2.1 — Load config from configs/config_9x9.json
import json

VARIANT = "n2"  # ← change to "n4" for 4-player training

CONFIG_PATH = f"{REPO_DIR}/configs/config_9x9.json"
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

# Merge variant-specific overrides into top level
variant = cfg["variants"][VARIANT]

N = variant["num_players"]
BOARD = cfg["board_size"]
WALLS = variant["max_walls_per_player"]
MAX_TURNS = cfg["mcts"]["max_rollout_depth"]

# Network
NUM_CHANNELS = cfg["network"]["num_channels"]
NUM_RES_BLOCKS = cfg["network"]["num_res_blocks"]
IN_CHANNELS = 3 * N + 3

# Action space
from src.env.quoridor_env_mp import compute_action_space_size
ACTION_SIZE = compute_action_space_size(BOARD)

# Training
NUM_ITERATIONS = cfg["training"]["num_iterations"]
GAMES_PER_ITER = cfg["training"]["games_per_iteration"]
MCTS_SIMS = cfg["mcts"]["num_simulations"]
BATCH_SIZE = cfg["training"]["batch_size"]
TRAIN_STEPS = cfg["training"]["training_epochs"]
REPLAY_BUFFER = cfg["training"]["replay_buffer_size"]
MAX_MOVES = cfg["training"]["max_game_moves"]
DISCOUNT = cfg["training"]["reward_decay"]
LR = cfg["training"]["learning_rate"]
WEIGHT_DECAY = cfg["training"]["weight_decay"]
EVAL_GAMES = variant["eval_games"]
EVAL_RANDOM = variant["eval_random_games"]

# Parallel self-play
NUM_WORKERS = cfg["parallel"]["num_workers"]
INFER_BATCH = cfg["parallel"]["inference_batch_size"]

print(f"Loaded: {CONFIG_PATH} [variant={VARIANT}]")
print(f"Config: N={N}, board={BOARD}×{BOARD}, actions={ACTION_SIZE}, walls={WALLS}")
print(f"Network: {NUM_RES_BLOCKS} res blocks, {NUM_CHANNELS} channels, {IN_CHANNELS} input planes")
print(f"Training: {NUM_ITERATIONS} iters, {GAMES_PER_ITER} games/iter, {MCTS_SIMS} sims")
print(f"Parallel: {NUM_WORKERS} workers, batch={INFER_BATCH}")

---
## 3. Validation
Quick smoke test before committing hours of GPU time.

In [ ]:
# 3.1 — Smoke test: env + tensor + model forward pass at 9×9
import numpy as np
from src.env.quoridor_env_mp import QuoridorEnvMP
from src.model.network_mp import QuoridorModelMP

env = QuoridorEnvMP(board_size=BOARD, num_players=N, max_turns=MAX_TURNS,
                    max_walls_per_player=WALLS)
state = env.reset()
tensor = env.state_to_tensor(state)

print(f"Tensor shape: {tensor.shape}  (expected: ({BOARD}, {BOARD}, {IN_CHANNELS}))")
assert tensor.shape == (BOARD, BOARD, IN_CHANNELS)

model = QuoridorModelMP(
    board_size=BOARD, action_space_size=ACTION_SIZE,
    in_channels=IN_CHANNELS, num_channels=NUM_CHANNELS,
    num_res_blocks=NUM_RES_BLOCKS, num_players=N,
    lr=LR, weight_decay=WEIGHT_DECAY,
)

policy, value = model.predict(tensor)
print(f"Policy shape: {policy.shape}  (expected: ({ACTION_SIZE},))")
print(f"Value shape:  {value.shape}  (expected: ({N},))")
assert policy.shape == (ACTION_SIZE,)
assert value.shape == (N,)

# Batch forward pass (what the GPU inference thread does)
batch = torch.from_numpy(tensor).float().permute(2, 0, 1).unsqueeze(0).repeat(INFER_BATCH, 1, 1, 1).to(model.device)
policies, values = model.predict_batch(batch)
print(f"Batch policy: {policies.shape}  (expected: ({INFER_BATCH}, {ACTION_SIZE}))")
print(f"Batch value:  {values.shape}  (expected: ({INFER_BATCH}, {N}))")

params = sum(p.numel() for p in model.network.parameters())
print(f"\nModel parameters: {params:,}")
print("\n✓ Smoke test passed — 9×9 pipeline is connected.")

In [ ]:
# 3.2 — Quick random game at 9×9 (no crash test)
env = QuoridorEnvMP(board_size=BOARD, num_players=N, max_turns=MAX_TURNS,
                    max_walls_per_player=WALLS)
state = env.reset()
moves = 0
while not state.game_over:
    actions = env.legal_actions(state)
    action = np.random.choice(actions)
    state = env.step(state, action)
    moves += 1

print(f"Random game finished in {moves} moves. Winner: P{state.winner}")
print("✓ No crashes during random play.")

---
## 4. Training
Re-run this cell after Colab disconnects — `resume=True` picks up from the last checkpoint.

In [ ]:
# 4 — Full 9×9 training (parallel self-play, GPU-batched)
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    force=True,
)

from src.env.quoridor_env_mp import QuoridorEnvMP
from src.model.network_mp import QuoridorModelMP
from src.mcts.training_mp import TrainingConfigMP, training_loop_mp

env = QuoridorEnvMP(board_size=BOARD, num_players=N, max_turns=MAX_TURNS,
                    max_walls_per_player=WALLS)

def make_model():
    return QuoridorModelMP(
        board_size=BOARD, action_space_size=ACTION_SIZE,
        in_channels=IN_CHANNELS, num_channels=NUM_CHANNELS,
        num_res_blocks=NUM_RES_BLOCKS, num_players=N,
        lr=LR, weight_decay=WEIGHT_DECAY, device="auto",
    )

model = make_model()

train_cfg = TrainingConfigMP(
    num_players=N,
    num_iterations=NUM_ITERATIONS,
    games_per_iteration=GAMES_PER_ITER,
    mcts_simulations=MCTS_SIMS,
    batch_size=BATCH_SIZE,
    train_steps_per_iter=TRAIN_STEPS,
    eval_games=EVAL_GAMES,
    eval_random_games=EVAL_RANDOM,
    accept_margin=0.05,
    max_game_moves=MAX_MOVES,
    discount=DISCOUNT,
    replay_buffer_size=REPLAY_BUFFER,
    mcts_dirichlet_epsilon=cfg.get('mcts', {}).get('dirichlet_epsilon', 0.25),
    # parallel self-play
    parallel_self_play=True,
    num_workers=NUM_WORKERS,
    inference_batch_size=INFER_BATCH,
    # geometry for spawned workers
    board_size=BOARD,
    max_walls_per_player=WALLS,
    max_turns=MAX_TURNS,
)

print(f"Starting training: {RUN_DIR}")
print(f"  Resume from checkpoint if available.")

training_loop_mp(
    env=env,
    model=model,
    make_model_fn=make_model,
    config=train_cfg,
    checkpoint_dir=RUN_DIR,
)


---
## 5. Monitor & Analyze Results

In [ ]:
# 5.1 — Training curves
import json
import matplotlib.pyplot as plt

meta_path = f"{RUN_DIR}/meta.json"

try:
    with open(meta_path) as f:
        meta = json.load(f)
    metrics = meta.get("history", [])
except FileNotFoundError:
    print("No meta.json yet — run training first.")
    metrics = []

if metrics:
    iters = [m['iteration'] for m in metrics]
    loss_p = [m.get('loss_policy', 0) for m in metrics]
    loss_v = [m.get('loss_value', 0) for m in metrics]
    wr = [m.get('win_rate_vs_random', 0) for m in metrics]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Quoridor 9×9 N=2 — Training Progress', fontsize=14)

    axes[0].plot(iters, loss_p, 'b-o', markersize=3)
    axes[0].set_title('Policy Loss')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Cross-Entropy')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iters, loss_v, 'r-o', markersize=3)
    axes[1].set_title('Value Loss')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('MSE')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(iters, [w * 100 for w in wr], 'g-o', markersize=3)
    axes[2].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
    axes[2].set_title('Win Rate vs Random')
    axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('Win Rate (%)')
    axes[2].set_ylim(0, 105)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{RUN_DIR}/training_curves.png", dpi=150)
    plt.show()

    print(f"\nLatest iteration: {iters[-1]}")
    print(f"Policy loss: {loss_p[-1]:.4f}")
    print(f"Value loss:  {loss_v[-1]:.4f}")
    print(f"Win rate:    {wr[-1]:.1%}")

In [ ]:
# 5.2 — Metrics table
if metrics:
    print(f"{'Iter':>4} | {'Loss_P':>8} | {'Loss_V':>8} | {'WR_Random':>10} | {'Accepted':>8}")
    print("-" * 55)
    for m in metrics:
        print(
            f"{m['iteration']:4d} | "
            f"{m.get('loss_policy', 0):8.4f} | "
            f"{m.get('loss_value', 0):8.4f} | "
            f"{m.get('win_rate_vs_random', 0):9.1%} | "
            f"{'✓' if m.get('model_accepted') else ''}"
        )

---
## 5.3 Run All Graph Scripts
Generate full dashboards and comparison figures (same scripts used for the project book).

In [ ]:
# 5.3 — Run all graph/plotting scripts
import subprocess, os

os.chdir(REPO_DIR)

scripts = [
    "scripts/plot_training.py",
    "scripts/plot_all_figures.py",
]

for script in scripts:
    if os.path.exists(script):
        print(f"\n{'='*60}")
        print(f"Running: {script}")
        print(f"{'='*60}")
        result = subprocess.run(
            ["python", script, RUN_DIR + "/meta.json"] if "plot_training" in script
            else ["python", script],
            capture_output=True, text=True,
            env={**os.environ, "PYTHONPATH": REPO_DIR}
        )
        if result.returncode == 0:
            print(f"✓ {script} completed")
            if result.stdout.strip():
                print(result.stdout[-500:])
        else:
            print(f"✗ {script} failed (exit {result.returncode})")
            print(result.stderr[-500:])
    else:
        print(f"⚠ {script} not found — skipping")

# Show generated figures
from IPython.display import Image, display
from pathlib import Path

fig_dirs = [
    Path(RUN_DIR) / "figures",
    Path(REPO_DIR) / "outputs",
]
for fig_dir in fig_dirs:
    if fig_dir.exists():
        for img in sorted(fig_dir.glob("*.png")):
            print(f"\n📊 {img.name}")
            display(Image(filename=str(img), width=900))

---
## 6. Export Best Model
Copy the final trained model to Drive and prepare for deployment.

In [ ]:
# 6.1 — Copy best/ship checkpoint
import shutil

best_src = f"{RUN_DIR}/best.pt"
ship_dst = f"{RUN_DIR}/ship.pt"

if os.path.exists(best_src):
    shutil.copy2(best_src, ship_dst)
    size_mb = os.path.getsize(ship_dst) / 1e6
    print(f"✓ Exported ship.pt ({size_mb:.1f} MB)")
    print(f"  Path: {ship_dst}")
else:
    print("No best.pt found — training may not have completed an accepted iteration yet.")

In [ ]:
# 6.2 — Download checkpoint to local machine (Colab only)
try:
    from google.colab import files
    if os.path.exists(ship_dst):
        files.download(ship_dst)
except ImportError:
    print("Not running on Colab — use Drive or scp to retrieve the model.")